In [1]:
import ee

ee.Authenticate()
ee.Initialize(project='avilight-483312')

# --- Time range: January 2014 through December 2025 ---
START_DATE = ee.Date('2014-01-01')
N_MONTHS = 144

# --- Google Drive export ---
DRIVE_FOLDER = 'Satellite Datasets'
EXPORT_NAME = 'province_rain_monthly_2014_2025'

# --- Philippine province boundaries ---
provinces = (
    ee.FeatureCollection('FAO/GAUL/2025/level2')
    .filter(ee.Filter.eq('GAUL0_NAME', 'Philippines'))
)

print('Province features:', provinces.size().getInfo())


def monthly_rain(month_index):
    """
    Create one IMERG rainfall image in mm/month.
    IMERG precipitation is in mm/hr, so:
    monthly mean rate × number of hours in that month = mm/month.
    """
    month_index = ee.Number(month_index)

    d0 = START_DATE.advance(month_index, 'month')
    d1 = d0.advance(1, 'month')

    monthly_collection = (
        ee.ImageCollection('NASA/GPM_L3/IMERG_V07')
        .filterDate(d0, d1)
        .select('precipitation')
    )

    hours_in_month = d1.difference(d0, 'day').multiply(24)

    rain_mm = (
        monthly_collection
        .mean()
        .multiply(hours_in_month)
        .rename('rain_mm')
    )

    # Province mean of this band is the spatial share of valid rain pixels.
    rain_valid = rain_mm.mask().rename('rain_valid')

    return (
        rain_mm.addBands(rain_valid)
        .set({
            'year': d0.get('year'),
            'month': d0.get('month'),
            'date': d0.format('YYYY-MM')
        })
    )


# Build monthly rainfall images for 2014-01 through 2025-12.
month_indices = ee.List.sequence(0, N_MONTHS - 1)

rain_monthly = ee.ImageCollection.fromImages(
    month_indices.map(monthly_rain)
)


def reduce_month_to_provinces(image):
    """Calculate mean monthly rainfall and valid-pixel share per province."""
    reduced = image.reduceRegions(
        collection=provinces,
        reducer=ee.Reducer.mean(),
        scale=5000,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'adm1': feature.get('GAUL1_NAME'),
            'province': feature.get('GAUL2_NAME'),
            'adm2_code': feature.get('GAUL2_CODE'),
            'year': image.get('year'),
            'month': image.get('month'),
            'rain_mm': feature.get('rain_mm'),
            'rain_valid': feature.get('rain_valid')
        })

    return reduced.map(format_row)


province_rain_fc = rain_monthly.map(
    reduce_month_to_provinces
).flatten()

province_rain_export = province_rain_fc.select([
    'adm1',
    'province',
    'adm2_code',
    'year',
    'month',
    'rain_mm',
    'rain_valid'
])

task = ee.batch.Export.table.toDrive(
    collection=province_rain_export,
    description=EXPORT_NAME,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_NAME,
    fileFormat='CSV'
)

task.start()

print('✅ Province-level monthly rainfall export started')
print('Task ID:', task.id)
print('Drive folder:', DRIVE_FOLDER)
print('Period: January 2014 to December 2025')

Province features: 85
✅ Province-level monthly rainfall export started
Task ID: SXXNYERMRD4TCAK56ZO2VEBO
Drive folder: Satellite Datasets
Period: January 2014 to December 2025


In [3]:
print(task.status())

{'state': 'COMPLETED', 'description': 'province_rain_monthly_2014_2025', 'priority': 100, 'creation_timestamp_ms': 1794717397233, 'update_timestamp_ms': 1794717579611, 'start_timestamp_ms': 1794717399196, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1-bU5d-UWd39u4OkyDb09gBu3DzCrayi9'], 'attempt': 1, 'batch_eecu_usage_seconds': 7257.0068359375, 'id': 'SXXNYERMRD4TCAK56ZO2VEBO', 'name': 'projects/avilight-483312/operations/SXXNYERMRD4TCAK56ZO2VEBO'}


In [4]:
import pandas as pd

df_rain = pd.read_csv('province_rain_monthly_2014_2025.csv')

print('Total rows:', len(df_rain))
print('Provinces:', df_rain['adm2_code'].nunique())
print('Months:', df_rain[['year', 'month']].drop_duplicates().shape[0])
print('Expected rows:', df_rain['adm2_code'].nunique() * 144)
print('Low-valid rows:', (df_rain['rain_valid'] < 0.3).sum())

df_rain.loc[df_rain['rain_valid'] < 0.3, 'rain_mm'] = pd.NA
df_rain.to_parquet('province_rain_monthly.parquet', index=False)

Total rows: 12240
Provinces: 85
Months: 144
Expected rows: 12240
Low-valid rows: 0
